In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:

from google.colab import files

uploaded = files.upload()


Saving loan_data.csv to loan_data.csv


In [3]:

import pandas as pd

filename = next(iter(uploaded))
df = pd.read_csv("loan_data.csv")

df.head()


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [4]:

print(df.shape)
print(df.columns)


(45000, 14)
Index(['person_age', 'person_gender', 'person_education', 'person_income',
       'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'credit_score', 'previous_loan_defaults_on_file', 'loan_status'],
      dtype='object')


In [5]:

print(df["loan_status"].value_counts())


loan_status
0    35000
1    10000
Name: count, dtype: int64


In [6]:

print(df.isnull().sum())


person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64


In [7]:

df = df.drop_duplicates()

print("Dataset shape:", df.shape)


Dataset shape: (45000, 14)


In [8]:

X = df.drop("loan_status", axis=1)
y = df["loan_status"]

print(X.head())
print(y.head())


   person_age person_gender person_education  person_income  person_emp_exp  \
0        22.0        female           Master        71948.0               0   
1        21.0        female      High School        12282.0               0   
2        25.0        female      High School        12438.0               3   
3        23.0        female         Bachelor        79753.0               0   
4        24.0          male           Master        66135.0               1   

  person_home_ownership  loan_amnt loan_intent  loan_int_rate  \
0                  RENT    35000.0    PERSONAL          16.02   
1                   OWN     1000.0   EDUCATION          11.14   
2              MORTGAGE     5500.0     MEDICAL          12.87   
3                  RENT    35000.0     MEDICAL          15.23   
4                  RENT    35000.0     MEDICAL          14.27   

   loan_percent_income  cb_person_cred_hist_length  credit_score  \
0                 0.49                         3.0           561  

In [9]:

X = pd.get_dummies(X, drop_first=True)

print(X.head())
print("Total features:", X.shape[1])


   person_age  person_income  person_emp_exp  loan_amnt  loan_int_rate  \
0        22.0        71948.0               0    35000.0          16.02   
1        21.0        12282.0               0     1000.0          11.14   
2        25.0        12438.0               3     5500.0          12.87   
3        23.0        79753.0               0    35000.0          15.23   
4        24.0        66135.0               1    35000.0          14.27   

   loan_percent_income  cb_person_cred_hist_length  credit_score  \
0                 0.49                         3.0           561   
1                 0.08                         2.0           504   
2                 0.44                         3.0           635   
3                 0.44                         2.0           675   
4                 0.53                         4.0           586   

   person_gender_male  person_education_Bachelor  ...  \
0               False                      False  ...   
1               False           

In [10]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


Training data: (36000, 22)
Testing data: (9000, 22)


In [11]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Scaling completed!")


Scaling completed!


In [12]:

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Model trained successfully!")


Model trained successfully!


In [13]:

from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred) * 100, "%")


Accuracy: 89.44444444444444 %


In [14]:

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.93      0.94      0.93      6990
           1       0.78      0.74      0.76      2010

    accuracy                           0.89      9000
   macro avg       0.85      0.84      0.85      9000
weighted avg       0.89      0.89      0.89      9000



In [15]:

!pip install gradio


In [16]:

import gradio as gr

def predict_loan(row_number):
    row_number = int(row_number)

    sample = X.iloc[[row_number]]

    sample = scaler.transform(sample)

    prediction = model.predict(sample)[0]

    if prediction == 1:
        return "Loan Approved"
    else:
        return "Loan Rejected"

app = gr.Interface(
    fn=predict_loan,
    inputs=gr.Number(
        label="Enter Row Number (0 to 44999)",
        value=0
    ),
    outputs=gr.Textbox(label="Loan Prediction"),
    title="Loan Approval Prediction",
    description="Enter a dataset row number to predict its loan status."
)

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://456937fc543aa93339.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:

!pip install flask


In [18]:

import os

os.makedirs("templates", exist_ok=True)


In [19]:

%%writefile templates/loan.html
<!DOCTYPE html>
<html>
<head>
    <title>Loan Approval Prediction</title>
</head>
<body>
    <h1>Loan Approval Prediction</h1>

    <form action="/predict" method="post">
        <label>Enter Row Number (0 to 44999):</label>
        <input type="number" name="row_number"
               min="0" max="44999" required>

        <button type="submit">Predict</button>
    </form>

    {% if result %}
        <h2>{{ result }}</h2>
    {% endif %}
</body>
</html>


Writing templates/loan.html


In [20]:

from flask import Flask, render_template, request

app = Flask(__name__)

@app.route("/")
def home():
    return render_template("loan.html", result=None)

@app.route("/predict", methods=["POST"])
def predict():
    row_number = int(request.form["row_number"])

    sample = X.iloc[[row_number]]
    sample = scaler.transform(sample)

    prediction = model.predict(sample)[0]

    if prediction == 1:
        result = "Loan Approved"
    else:
        result = "Loan Rejected"

    return render_template("loan.html", result=result)

print("Flask application created successfully!")


Flask application created successfully!


In [21]:

import threading

thread = threading.Thread(
    target=lambda: app.run(
        host="0.0.0.0",
        port=5000,
        use_reloader=False
    )
)

thread.start()


In [22]:

from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(5000)")
print(url)


https://5000-m-s-kkb-usc1c0-2wggmpnof7eds-c.us-central1-0.prod.colab.dev
